In [11]:
import os
from dotenv import load_dotenv

# Carga de variables de entorno (o soporte para Google Colab Secrets)
load_dotenv()

if "google.colab" in str(get_ipython()):
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY en las variables de entorno"

# Configuración de observabilidad con LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Nexus-MultiTenant-RAG"

from langchain_groq import ChatGroq
from langsmith import traceable

# Instanciación del LLM optimizado para velocidad en producción
MODELO_LLM = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
llm = ChatGroq(model=MODELO_LLM, temperature=0.1)

print("✓ Entorno configurado. LangSmith activado y LLM instanciado.")

✓ Entorno configurado. LangSmith activado y LLM instanciado.


In [12]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Inicialización de embeddings locales usando la integración moderna
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Catálogo simulado con partición comercial por franquicia (Multi-Tenant)
documentos = [
    {
        "text": "Cyberpunk 2077. RPG de acción en mundo abierto ambientado en Night City.",
        "metadata": {"tenant_id": "nexus_chile", "title": "Cyberpunk 2077", "price_clp": 35000, "stock": 15, "genres": "RPG, Acción"}
    },
    {
        "text": "Elden Ring. Juego de rol de acción y fantasía oscura desarrollado por FromSoftware.",
        "metadata": {"tenant_id": "nexus_chile", "title": "Elden Ring", "price_clp": 45000, "stock": 5, "genres": "RPG, Fantasía"}
    },
    {
        "text": "Stardew Valley. Simulador de granja y vida rural pacífica.",
        "metadata": {"tenant_id": "nexus_global_usd", "title": "Stardew Valley", "price_usd": 15, "stock": 100, "genres": "Simulación, Casual"}
    }
]

texts = [doc["text"] for doc in documentos]
metadatas = [doc["metadata"] for doc in documentos]

# Vector store con soporte para aislamiento por metadatos
vectorstore = Chroma.from_texts(
    texts=texts,
    metadatas=metadatas,
    embedding=embeddings,
    collection_name="nexus_digital_store"
)

print("✓ Base de datos vectorial ChromaDB sincronizada para entornos Multi-Tenant.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Base de datos vectorial ChromaDB sincronizada para entornos Multi-Tenant.


In [19]:
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langsmith import traceable
from langchain_community.utilities import WikipediaAPIWrapper

# 1. Definición del Estado Híbrido
class AssistantState(TypedDict):
    query: str
    tenant_id: str
    context: str                 # Contexto Interno (ChromaDB)
    external_context: str        # Contexto Externo (Wikipedia)
    response: str
    messages: Annotated[list[BaseMessage], add_messages]

# Instanciamos el conector de Wikipedia
wikipedia = WikipediaAPIWrapper(lang="es", top_k_results=1, doc_content_chars_max=400)

# 2. Prompting Avanzado Híbrido (Zero-Shot + Few-Shot + CoT)
SYSTEM_PROMPT = """Eres NexusBot, el Asistente Experto en Ventas de PC Gaming.
Tu objetivo es vender basándote en el catálogo interno, y enriquecer la charla usando el contexto externo.

REGLAS ESTRICTAS:
1. Si el stock es 0, advierte claramente que no hay unidades.
2. Muestra los precios en la moneda del catálogo local.
3. Utiliza la información de Wikipedia (Dato curioso) para entusiasmar al cliente con la trama.
4. Piensa paso a paso (Chain-of-Thought) antes de dar la respuesta final.

EJEMPLO DE COMPORTAMIENTO:
Usuario: "¿Tienen Cyberpunk?"
Razonamiento: Busco "Cyberpunk" en el catálogo interno. Hay 15 unidades a 35000 CLP. Reviso Wikipedia: trata sobre mercenarios en Night City.
Respuesta final: Sí, tenemos **Cyberpunk 2077** en stock (15 unidades) a $35.000 CLP. Como dato curioso, ¡este juego te sumergirá en el peligroso mundo de Night City como mercenario!

CATÁLOGO DISPONIBLE (Interno):
{context}

DATO CURIOSO DEL JUEGO (Externo - Wikipedia):
{external_context}
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="messages")
])

# 3. Nodo de Recuperación Híbrida (Interno + Externo)
@traceable(name="nexus_retrieval_hybrid")
def retrieve(state: AssistantState) -> dict:
    # A. Recuperación Interna en ChromaDB
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 3, "filter": {"tenant_id": state["tenant_id"]}}
    )
    docs = retriever.invoke(state["query"])
    
    context_parts = []
    juegos_encontrados = []
    
    for i, doc in enumerate(docs, 1):
        md = doc.metadata
        precio = f"${md.get('price_clp', 'N/A')} CLP" if "price_clp" in md else f"${md.get('price_usd', 'N/A')} USD"
        context_parts.append(f"[{i}] {md.get('title')} - {precio} | Géneros: {md.get('genres')} | Stock: {md.get('stock')}")
        juegos_encontrados.append(md.get('title'))
        
    contexto_interno = "\n".join(context_parts) if context_parts else "Sin resultados en el catálogo."
    
    # B. Recuperación Externa en Wikipedia
    contexto_externo = "Sin datos externos adicionales."
    if juegos_encontrados:
        try:
            # Consulta a Wikipedia usando el título del primer juego recuperado
            contexto_externo = wikipedia.run(juegos_encontrados[0] + " videojuego")
        except Exception:
            pass # Si falla la API externa, el RAG no se rompe
            
    return {
        "context": contexto_interno, 
        "external_context": contexto_externo
    }

# 4. Nodo de Generación
@traceable(name="nexus_generation")
def generate(state: AssistantState) -> dict:
    cadena = prompt_template | llm | StrOutputParser()
    respuesta = cadena.invoke({
        "context": state["context"],
        "external_context": state.get("external_context", ""),
        "messages": state["messages"]
    })
    
    return {"response": respuesta, "messages": [AIMessage(content=respuesta)]}

# 5. Ensamblado del Grafo Híbrido
workflow = StateGraph(AssistantState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

memory = MemorySaver()
app_ecommerce = workflow.compile(checkpointer=memory)

print("✓ Grafo compilado: RAG Híbrido (Interno + Externo) con Wikipedia.")

✓ Grafo compilado: RAG Híbrido (Interno + Externo) con Wikipedia.


In [20]:
import time

# Configuramos un ID de sesión único para la memoria
config_sesion = {"configurable": {"thread_id": "sesion_nexus_001"}}

def chat_stream(query: str, tenant_id: str):
    print(f"\n👤 Usuario: {query}")
    print("🤖 NexusBot: ", end="", flush=True)
    
    inputs = {
        "query": query,
        "tenant_id": tenant_id,
        "messages": [HumanMessage(content=query)]
    }
    
    # Streaming de la salida en tiempo real
    respuesta_completa = ""
    for event in app_ecommerce.stream(inputs, config=config_sesion, stream_mode="updates"):
        for node_name, node_data in event.items():
            if node_name == "generate":
                # Simulamos streaming visual (Groq es muy rápido)
                for char in node_data["response"]:
                    print(char, end="", flush=True)
                    time.sleep(0.005)
                respuesta_completa = node_data["response"]
                contexto_usado = node_data.get("context", "")
    print("\n" + "-"*60)
    return respuesta_completa

# Turno 1: Recuperación inicial
res_1 = chat_stream("¿Qué juegos de rol tienen y cuánto valen?", "nexus_chile")

# Turno 2: Prueba de memoria conversacional (El bot debe recordar el contexto y tu intención)
res_2 = chat_stream("¿Cuál de esos es el más barato?", "nexus_chile")


👤 Usuario: ¿Qué juegos de rol tienen y cuánto valen?
🤖 NexusBot: **Razonamiento paso a paso**  
1. Reviso el catálogo interno y filtro los juegos cuyo género incluya “RPG”.  
2. En el catálogo solo aparece **Elden Ring**, que está clasificado como RPG y fantasía.  
3. Verifico el precio y el stock: $45.000 CLP por unidad, con 5 unidades disponibles.  
4. Consulto la información externa de Wikipedia para añadir un dato curioso, pero el catálogo indica que no hay datos externos adicionales disponibles para este juego.  

**Respuesta final**  

Actualmente, en nuestro inventario disponemos del siguiente juego de rol:

- **Elden Ring** – Precio: **$45.000 CLP** – Stock: **5 unidades**  

*Dato curioso:* En este momento no contamos con información adicional de Wikipedia sobre Elden Ring.  

Si te interesa adquirirlo o deseas más detalles, ¡avísame!
------------------------------------------------------------

👤 Usuario: ¿Cuál de esos es el más barato?
🤖 NexusBot: **Razonamiento paso a paso

In [21]:
import json
import re

@traceable(name="eval_faithfulness")
def evaluar_fidelidad(query: str, contexto: str, respuesta: str) -> float:
    """Mide si la respuesta generada está respaldada estrictamente por el contexto (1.0) o alucina (0.0)"""
    prompt_eval = f"""Evalúa si la respuesta es fiel al contexto proporcionado.
    Consulta: {query}
    Contexto: {contexto}
    Respuesta: {respuesta}
    
    Responde UNICAMENTE con un JSON válido usando esta estructura:
    {{"puntaje": <numero de 0.0 a 1.0>, "razon": "<explicacion breve>"}}
    """
    res = llm.invoke(prompt_eval).content
    try:
        match = re.search(r'\{.*\}', res.strip(), re.DOTALL)
        datos = json.loads(match.group(0)) if match else json.loads(res)
        return float(datos.get("puntaje", 0.0))
    except Exception as e:
        return 0.0

@traceable(name="eval_relevance")
def evaluar_relevancia(query: str, respuesta: str) -> float:
    """Mide si la respuesta contesta directamente la pregunta del usuario sin desvíos (0.0 a 1.0)"""
    prompt_eval = f"""Evalúa qué tan relevante es la respuesta para la consulta.
    Consulta: {query}
    Respuesta: {respuesta}
    
    Responde UNICAMENTE con un JSON válido usando esta estructura:
    {{"puntaje": <numero de 0.0 a 1.0>, "razon": "<explicacion breve>"}}
    """
    res = llm.invoke(prompt_eval).content
    try:
        match = re.search(r'\{.*\}', res.strip(), re.DOTALL)
        datos = json.loads(match.group(0)) if match else json.loads(res)
        return float(datos.get("puntaje", 0.0))
    except Exception as e:
        return 0.0

# Extracción de la memoria para evaluación
estado_actual = app_ecommerce.get_state(config_sesion).values
ultimo_contexto = estado_actual.get("context", "Sin contexto")
ultima_pregunta = "¿Cuál de esos es el más barato?"

print("📊 EJECUCIÓN DE MÉTRICAS RAG (LangSmith Telemetry)")
print("=" * 50)

fidelidad = evaluar_fidelidad(ultima_pregunta, ultimo_contexto, res_2)
relevancia = evaluar_relevancia(ultima_pregunta, res_2)

print(f"🔹 Faithfulness (Fidelidad al Contexto): {fidelidad:.2f}/1.0")
print(f"🔹 Answer Relevancy (Relevancia a la Pregunta): {relevancia:.2f}/1.0")

if fidelidad >= 0.8 and relevancia >= 0.8:
    print("\n✅ Evaluación Exitosa: El pipeline genera respuestas precisas, contextualizadas y sin alucinaciones.")
else:
    print("\n⚠️ Advertencia: Se detectaron posibles alucinaciones o desvíos. Se requiere ajuste de prompts.")

📊 EJECUCIÓN DE MÉTRICAS RAG (LangSmith Telemetry)
🔹 Faithfulness (Fidelidad al Contexto): 1.00/1.0
🔹 Answer Relevancy (Relevancia a la Pregunta): 0.95/1.0

✅ Evaluación Exitosa: El pipeline genera respuestas precisas, contextualizadas y sin alucinaciones.
